# 🩺 AnKing-v11 Dataset Explorer & Anatomy
Welcome to the interactive exploration notebook for the **AnKing Overhaul v11** medical knowledge base.

### What is AnKing-v11?
The AnKing deck is the de-facto gold standard flashcard suite used by medical students preparing for **USMLE Step 1** and **Step 2 CK**. It aggregates and refines the most famous community decks:
- **Zanki Step Decks**: Organ system pathology & physiology (Cardio, Renal, Pulm, GI, etc.)
- **Zanki Pharmacology**: Drug mechanisms, side effects, and clinical indications
- **Lolnotacop**: Comprehensive medical microbiology & antimicrobial therapy
- **Cheesy Dorian / Zanki Step 2**: Clinical diagnosis, next best steps, and management

### In this Notebook:
1. **Deck Architecture & Inventory**: Scan and quantify cards across Step 1 and Step 2.
2. **Note Anatomy**: Inspect how cards, cloze deletions, and explanations are structured.
3. **Curriculum & Resource Tags**: Discover how cards map to First Aid, UWorld, Pathoma, and Sketchy.
4. **Interactive Search**: Query the dataset directly by symptom, drug, or disease.
5. **Cloze Parser**: Test prompt generation and answer extraction.

In [1]:
from pathlib import Path
import re
from collections import Counter
import pandas as pd
import html

# Detect directory path (supporting running from notebook or root)
candidates = [
    Path("../data/AnKing-v11"),
    Path("lab/data/AnKing-v11"),
    Path("/home/semere/sources/usmle-study-helper/lab/data/AnKing-v11")
]

DATA_DIR = next((p for p in candidates if p.exists()), None)

if DATA_DIR is None:
    raise FileNotFoundError("Could not find AnKing-v11 directory! Ensure lab/data/AnKing-v11 exists.")

print(f"✅ AnKing Dataset Root: {DATA_DIR.resolve()}")
md_files = list(DATA_DIR.glob("**/*.md"))
# Exclude the root README.md
card_files = [p for p in md_files if p.name != "README.md"]
print(f"📊 Total Markdown Cards Discovered: {len(card_files):,}")

✅ AnKing Dataset Root: /home/semere/sources/usmle-study-helper/lab/data/AnKing-v11
📊 Total Markdown Cards Discovered: 34,712


## 1. Deck Hierarchy & Card Counts
Let's analyze the distribution of cards across **Step 1** (Pre-clinical sciences) and **Step 2** (Clinical medicine), broken down by sub-disciplines.

In [3]:
# Categorize cards by major category and subdeck
stats = []

for p in card_files:
    rel_parts = p.relative_to(DATA_DIR).parts
    top_level = rel_parts[0] if len(rel_parts) > 1 else "Root"
    subdeck = rel_parts[1] if len(rel_parts) > 2 else "General"
    stats.append({"Exam": top_level, "Subdeck": subdeck, "Path": p})

df_stats = pd.DataFrame(stats)

# Summary by major category
summary = df_stats.groupby(["Exam", "Subdeck"]).size().reset_index(name="Card Count")
summary = summary.sort_values(by="Card Count", ascending=False).reset_index(drop=True)

# Display table (with optional styling if jinja2 is installed)
try:
    display(summary.style.background_gradient(subset=["Card Count"], cmap="Blues"))
except Exception:
    display(summary)

,Exam,Subdeck,Card Count
0,Step 1,Zanki Step Decks,16528
1,Step 2,Cheesy Dorian (M3),8731
2,Step 1,Lolnotacop,4390
3,Step 1,Zanki Pharmacology,3293
4,Step 2,MedicalArk,863
5,Step 2,Zanki Step 2,781
6,Step 2,General,124
7,AnKing Note Types,AnKingOverhaul,1
8,AnKing Note Types,IO-one by one,1


## 2. Anatomy of an AnKing Markdown Card
Each file contains structured metadata, the prompt text with Cloze syntax (`{{c1::answer}}`), and extra clinical context.

Let's write a dedicated parser function to extract:
- `nid`: Note unique identifier
- `model`: Note type (e.g. `AnKingOverhaul`)
- `tags`: List of hierarchical tags
- `text`: Prompt text with cloze markers
- `extra`: Clinical explanations, differentials, and mechanisms

In [ ]:
def parse_anki_md(file_path: Path):
    """Parse a ki-decompiled Anki markdown file into structured fields."""
    try:
        content = file_path.read_text(encoding="utf-8")
    except Exception as e:
        return None

    data = {
        "path": str(file_path),
        "nid": None,
        "model": None,
        "tags": [],
        "text": "",
        "extra": ""
    }

    # Extract header metadata
    nid_m = re.search(r"^nid:\s*(\d+)", content, re.M)
    if nid_m:
        data["nid"] = int(nid_m.group(1))

    model_m = re.search(r"^model:\s*(.+)", content, re.M)
    if model_m:
        data["model"] = model_m.group(1).strip()

    tags_m = re.search(r"^tags:\s*(.+)", content, re.M)
    if tags_m:
        raw_tags = tags_m.group(1).split(",")
        data["tags"] = [t.strip() for t in raw_tags if t.strip()]

    # Extract ### Text section
    text_m = re.search(r"### Text\s*\n(.*?)(?=\n### |\Z)", content, re.DOTALL)
    if text_m:
        data["text"] = text_m.group(1).strip()

    # Extract ### Extra section
    extra_m = re.search(r"### Extra\s*\n(.*?)(?=\n### |\Z)", content, re.DOTALL)
    if extra_m:
        data["extra"] = extra_m.group(1).strip()

    return data

In [9]:

# Preview a sample parsed card
sample_card = parse_anki_md(card_files[100])
print(f"ID: {sample_card['nid']}")
print(f"Model: {sample_card['model']}")
print(f"Tags ({len(sample_card['tags'])} tags): {sample_card['tags'][:3]}...")
print("\n--- PROMPT (TEXT) ---")
print(sample_card['text'][:200])
print("\n--- EXTRA EXPLANATION ---")
print(sample_card['extra'][:200])

ID: 1637500477613
Model: AnKingOverhaul
Tags (4 tags): ['#AK_Original_Decks::Step_2', '#AK_Step2_v11::!Shelf::ObGyn::no_dupes::only_step2', '#AK_Step2_v11::#B&B::06_ObGyn::Obstetrics::16_Preterm']...

--- PROMPT (TEXT) ---
What is the first line <b>tocolytic</b> for <b>preterm labor</b>?
{{c1::Indomethacin}}

--- EXTRA EXPLANATION ---
### Lecture Notes


## 3. Tag Taxonomy & Resource Mapping
AnKing is famous for tagging every card according to primary medical school curricula:
- `#FirstAid`: Organ systems & subtopics in First Aid for the USMLE Step 1
- `#UWorld`: Question IDs from UWorld QBank
- `#Pathoma`: Pathology chapters (Fundamentals, Neoplasia, Valvular Heart Disease, etc.)
- `#SketchyMicro` & `#SketchyPharm`: Visual mnemonic scenes
- `^HighYield`: Cards flagged as essential for passing and scoring 240+

Let's sample 1,000 cards and analyze which resources are represented most frequently.

In [10]:
# Sample 2,000 cards to map out resource categories
sample_cards = [parse_anki_md(f) for f in card_files[::17][:2000]]

resource_counts = Counter()
high_yield_topics = Counter()

for card in sample_cards:
    if not card:
        continue
    for t in card["tags"]:
        # Identify broad curriculum source
        if "FirstAid" in t:
            resource_counts["First Aid"] += 1
            # Extract First Aid chapter if possible
            fa_match = re.search(r"FirstAid::(?:\d+_)?([^:]+)", t)
            if fa_match:
                high_yield_topics[fa_match.group(1)] += 1
        elif "UWorld" in t:
            resource_counts["UWorld"] += 1
        elif "Pathoma" in t:
            resource_counts["Pathoma"] += 1
        elif "Sketchy" in t:
            resource_counts["Sketchy"] += 1
        elif "Physeo" in t:
            resource_counts["Physeo"] += 1
        elif "B&B" in t or "Boards" in t:
            resource_counts["Boards & Beyond"] += 1
        elif "HighYield" in t:
            resource_counts["High-Yield Tagged"] += 1

print("📚 Resource Breakdown in Sampled Cards:")
for res, count in resource_counts.most_common():
    print(f"  • {res:20s}: {count:,} cards")

print("\n🏥 Top High-Yield Organ Systems (from First Aid tags):")
for topic, count in high_yield_topics.most_common(8):
    print(f"  • {topic:25s}: {count:,} cards")

📚 Resource Breakdown in Sampled Cards:
  • First Aid           : 2,747 cards
  • Boards & Beyond     : 1,681 cards
  • Physeo              : 1,382 cards
  • Sketchy             : 1,009 cards
  • High-Yield Tagged   : 996 cards
  • UWorld              : 650 cards
  • Pathoma             : 559 cards

🏥 Top High-Yield Organ Systems (from First Aid tags):
  • Microbiology             : 488 cards
  • Neuro_&_Special_Senses   : 273 cards
  • Hematology_Oncology      : 217 cards
  • Biochemistry             : 215 cards
  • Cardiovascular           : 187 cards
  • Endocrine                : 182 cards
  • Repro                    : 176 cards
  • Musculoskeletal_Skin_Connective_Tissue: 160 cards


## 4. Cloze Deletion & Clean Text Rendering
Anki cards use the cloze format:
- `{{c1::Aortic stenosis}}` ➔ In test mode, this appears as `[...]`. In answer mode, this is highlighted as `Aortic stenosis`.
- `{{c1::Cervicitis::condition}}` ➔ Cloze with a hint (`[condition]`).

Cards also contain raw HTML (`<div>`, `<span>`, `<br>`, `<b>`). To make our app snappy and clean, we strip HTML and extract:
1. `prompt`: The question prompt with `[...]`
2. `answers`: The hidden answers
3. `clean_text`: Uncloaked plain text for full-text indexing

In [11]:
def strip_html(raw_html: str) -> str:
    """Remove HTML tags and decode entities."""
    if not raw_html:
        return ""
    clean = re.sub(r"<br\s*/?>", " ", raw_html, flags=re.I)
    clean = re.sub(r"<[^>]+>", "", clean)
    clean = html.unescape(clean)
    return re.sub(r"\s+", " ", clean).strip()

def render_cloze(raw_text: str, target_cloze: int = 1, show_answer: bool = False) -> str:
    """Render cloze deletions for test mode or answer mode."""
    def replacer(match):
        cloze_num = int(match.group(1))
        content = match.group(2)
        parts = content.split("::", 1)
        answer = parts[0]
        hint = parts[1] if len(parts) > 1 else "..."

        if cloze_num == target_cloze:
            return f"[{answer.upper()}]" if show_answer else f"[{hint}]"
        return answer  # Non-target clozes are shown as plain text

    # Pattern matches {{c1::answer}} or {{c1::answer::hint}}
    rendered = re.sub(r"\{\{c(\d+)::(.*?)\}\}", replacer, raw_text)
    return strip_html(rendered)

# Demonstration on a raw sample
demo_text = "<div><b>{{c1::Aortic stenosis::condition}}</b> presents with a {{c2::crescendo-decrescendo}} murmur.</div>"
print("Raw input HTML:")
print(" ", demo_text)
print("\nTest Mode (Cloze 1 Hidden):")
print(" ", render_cloze(demo_text, target_cloze=1, show_answer=False))
print("\nRevealed Mode (Cloze 1 Shown):")
print(" ", render_cloze(demo_text, target_cloze=1, show_answer=True))

Raw input HTML:
  <div><b>{{c1::Aortic stenosis::condition}}</b> presents with a {{c2::crescendo-decrescendo}} murmur.</div>

Test Mode (Cloze 1 Hidden):
  [condition] presents with a crescendo-decrescendo murmur.

Revealed Mode (Cloze 1 Shown):
  [AORTIC STENOSIS] presents with a crescendo-decrescendo murmur.


## 5. Interactive Concept & Disease Search
You can use this interactive search function to look up **any disease, drug, bug, or finding** across the entire 34,700-card AnKing library!

Try searching for:
- `"pericarditis"`
- `"Kawasaki"`
- `"metformin"`
- `"Cervicitis"`
- `"Aortic dissection"`

In [14]:
def search_anking_cards(query: str, max_results: int = 5):
    """Search the AnKing cards for a keyword in text or tags."""
    query_lower = query.lower()
    matches = []

    for file_path in card_files:
        card = parse_anki_md(file_path)
        if not card:
            continue

        clean_text = strip_html(card["text"]).lower()
        clean_extra = strip_html(card["extra"]).lower()
        tags_str = " ".join(card["tags"]).lower()

        if query_lower in clean_text or query_lower in clean_extra or query_lower in tags_str:
            matches.append(card)
            if len(matches) >= max_results:
                break

    print(f"🔎 Found {len(matches)} matches for: '{query}'\n")

    for i, c in enumerate(matches, 1):
        prompt = render_cloze(c["text"], target_cloze=1, show_answer=False)
        answer = render_cloze(c["text"], target_cloze=1, show_answer=True)
        extra = strip_html(c["extra"])
        tags = [t.split("::")[-1] for t in c["tags"][:3]]

        print(f"--- [Card #{i}] (ID: {c['nid']}) ---")
        print(f"📌 Prompt: {prompt}")
        print(f"💡 Answer: {answer}")
        if extra:
            print(f"📖 Extra:  {extra[:160]}...")
        print(f"🏷️  Tags:   {', '.join(tags)}")
        print()

# Test search query
search_anking_cards("HIV", max_results=20)

🔎 Found 20 matches for: 'HIV'

--- [Card #1] (ID: 1514672868031) ---
📌 Prompt: Drugs like Dolutegravir, Elvitegravir, and Raltegravir reversibly inhibit [...]
💡 Answer: Drugs like Dolutegravir, Elvitegravir, and Raltegravir reversibly inhibit [HIV INTEGRASE]
📖 Extra:  - These drugs prevent the newly reverse-transcribed dsDNA from being integrated into the host genome...
🏷️  Tags:   Zanki_Pharmacology, !DELETE, 07_HIV_Drugs

--- [Card #2] (ID: 1515560045767) ---
📌 Prompt: The toxicities that result from NRTIs are based in part by the inhibition of [...] DNA synthesis
💡 Answer: The toxicities that result from NRTIs are based in part by the inhibition of [MITOCHONDRIAL] DNA synthesis
🏷️  Tags:   Zanki_Pharmacology, !DELETE, 07_HIV_Drugs

--- [Card #3] (ID: 1515556520483) ---
📌 Prompt: NNRTIs can cause {{c1::Stevens-Johnson Syndrome::Dermatologic Syndrome}} as a side effect
💡 Answer: NNRTIs can cause {{c1::Stevens-Johnson Syndrome::Dermatologic Syndrome}} as a side effect
🏷️  Tags:   Zanki